### **Método de Uniformización para una CMTC**

Este notebook contiene la resolución de los ejercicios 3 y 4.2 de la actividad 6.

3.- (Ejercicio para programar) Este último teorema permite aproximar $P(t)$ usando los primeros $M$ términos de la serie infinita. Se obtienen buenos resultados si se elige:

$$M \approx máx \{rt + 5\sqrt{rt}, 20\}$$

3.1.-  Use esta propuesta para calcular $P(0.5)$, $P(1)$ y $P(5)$ para la matriz $R$ del ejercicio 1.

Definimos la matriz de tasas $R$ del ejercicio 1, calculamos el valor máximo $r$ y la matriz de transición discreta $\hat{P}$.

In [29]:
import numpy as np
import math

In [30]:
R=np.array([[0, 2, 3, 0], [4, 0, 2, 0], [0, 2, 0, 2], [1, 0, 3, 0]])
print(R)

[[0 2 3 0]
 [4 0 2 0]
 [0 2 0 2]
 [1 0 3 0]]


In [31]:
r0=0+2+3+0
r1=4+0+2+0
r2=0+2+0+2
r3=1+0+3+0
r_i= [r0,r1,r2,r3]
print("r_i=",r_i)
r=max(r_i)
print("El valor de r es:",r)

r_i= [5, 6, 4, 4]
El valor de r es: 6


In [32]:
P_gorro=np.zeros((4,4))
for i in range(4):
    for j in range(4):
        if i == j:
            P_gorro[i,j]=1-(r_i[i]/r)
        else:
            P_gorro[i,j]=R[i,j]/r
print("La matriz P_gorro es:")
print(np.round(P_gorro,4))

La matriz P_gorro es:
[[0.1667 0.3333 0.5    0.    ]
 [0.6667 0.     0.3333 0.    ]
 [0.     0.3333 0.3333 0.3333]
 [0.1667 0.     0.5    0.3333]]


Función para calcular $P(t)$

Se crea la función utilizando la sumatoria de Poisson hasta el valor $M$ calculado

In [33]:
def Pejer4(t):
    rt=r*t
    M=int(max(rt+5*math.sqrt(rt),20))
    P_t=np.zeros((4,4))
    P_gorro_k = np.eye(4)
    for k in range(M + 1):
        prob_poisson=math.exp(-rt)*((rt**k)/math.factorial(k))
        P_t=P_t+(prob_poisson*P_gorro_k)
        P_gorro_k=np.dot(P_gorro_k, P_gorro)
    return P_t, M
P_05,M_05=Pejer4(0.5)
P_1, M_1=Pejer4(1.0)
P_5, M_5=Pejer4(5.0)
print(f"P(0.5) con M={M_05}:", np.round(P_05, 4),)
print(f"P(1) con M={M_1}:", np.round(P_1, 4),)
print(f"P(5) con M={M_5}:", np.round(P_5, 4))

P(0.5) con M=20: [[0.2506 0.217  0.3867 0.1458]
 [0.2531 0.2384 0.3744 0.1341]
 [0.1691 0.1936 0.4203 0.217 ]
 [0.158  0.1574 0.3983 0.2862]]
P(1) con M=20: [[0.2062 0.2039 0.3987 0.1912]
 [0.2083 0.2053 0.3979 0.1885]
 [0.1968 0.1984 0.401  0.2039]
 [0.192  0.194  0.4015 0.2125]]
P(5) con M=57: [[0.2 0.2 0.4 0.2]
 [0.2 0.2 0.4 0.2]
 [0.2 0.2 0.4 0.2]
 [0.2 0.2 0.4 0.2]]


2.- ¿Se verifica la ecuación de Chapman-Kolmogorov $P(1)=P(0.5)P(0.5)$?

In [34]:

P_mult=np.dot(P_05, P_05)
print("Matriz P(1) directa:")
print(np.round(P_1,4),)
print("Matriz P(0.5)*P(0.5):")
print(np.round(P_mult, 4))

Matriz P(1) directa:
[[0.2062 0.2039 0.3987 0.1912]
 [0.2083 0.2053 0.3979 0.1885]
 [0.1968 0.1984 0.401  0.2039]
 [0.192  0.194  0.4015 0.2125]]
Matriz P(0.5)*P(0.5):
[[0.2062 0.2039 0.3987 0.1912]
 [0.2083 0.2053 0.3979 0.1885]
 [0.1968 0.1984 0.401  0.2039]
 [0.192  0.194  0.4015 0.2125]]


Como podemos observar en los resultados, los valores de la matriz $P(1)$ calculada directamente y los del producto matricial $P(0.5) \times P(0.5)$ coinciden. Por lo tanto, se verifica la ecuación de Chapman-Kolmogorov.

4.- Teorema (Cotas de error para $P(t)$): Para un $t \geq 0$ fijo, sea

$$
P^M(t) = [p_{i,j}^M(t)]
= \sum_{k=0}^{M} e^{-rt}\frac{(rt)^k}{k!}\hat{P}^k
$$

entonces $ |p_{i,j}(t)-p_{i,j}^M(t)| \leq \sum_{k=M+1}^{\infty} e^{-rt}\frac{(rt)^k}{k!}$ para todo $1 \leq i,j \leq N$.

2.- (Ejercicio para programar) Este teorema se puede usar así. Suponga que se desea calcular $P(t)$ con una tolerancia $\epsilon$. Elija $M$ tal que
$$\sum_{k=M+1}^{\infty} e^{-rt} \frac{(rt)^k}{k!} \leq \epsilon$$

Y se puede implementar de acuerdo al siguiente algoritmo de uniformización para $P(t)$:

1. Dados $R$, $t$, $0 < \epsilon < 1$.
2. Calcular $r$ usando la igualdad en la definición.
3. Calcular $\hat{P}$.
4. $A = \hat{P}$; $B = e^{-rt}I$; $c = e^{-rt}$; sum = $c$; $k = 1$
5. Mientras sum $< 1 - \epsilon$ hacer:
   * $c = c \cdot (rt)/k$
   * $B = B + cA$
   * $A = A\hat{P}$
   * $\text{sum} = \text{sum} + c$
   * $k = k + 1$
6. $B$ está a $\epsilon$ de $P(t)$.


Repita el ejercicio 3 aplicando este algoritmo con una tolerancia $\epsilon = 0.00001$ (indique el valor correspondiente de $M$ en cada caso). Compare los resultados.

In [35]:
def algoritmo(R,t,epsilon=0.00001):
    r=np.max(np.sum(R,axis=1))
    N=R.shape[0]
    P_gorro=np.zeros((N,N))
    for i in range(N):
        ri=np.sum(R[i,:])
        for j in range(N):
            if i == j:
                P_gorro[i,j]=1-(ri/r)
            else:
                P_gorro[i,j]=R[i,j]/r
    rt=r*t
    A=np.copy(P_gorro)
    B=math.exp(-rt) * np.eye(N)
    c=math.exp(-rt)
    suma=c
    k=1
    while suma < (1-epsilon):
        c=c*(rt/k)
        B=B+(c*A)
        A=np.dot(A,P_gorro)
        suma=suma+c
        k=k+1
    Mexac=k-1
    return B, Mexac

In [36]:
P_05_eps, M_05_eps=algoritmo(R,0.5)
P_1_eps, M_1_eps=algoritmo(R,1.0)
P_5_eps, M_5_eps=algoritmo(R,5.0)
print(f"Para t=0.5: el M de la formula es={M_05} y el M del algoritmo es={M_05_eps}")
print(f"Para t=1: el M de la formula es={M_1}, y el M del algoritmo es={M_1_eps}")
print(f"Para t=5: el M de la formula es={M_5}, y el M del algoritmo es={M_5_eps}")
print("Matriz P(0.5) con el algoritmo:")
print(np.round(P_05_eps, 4))
print("Matriz P(1) con el algoritmo:")
print(np.round(P_1_eps, 4))
print("Matriz P(5) con el algoritmo:")
print(np.round(P_5_eps, 4))

Para t=0.5: el M de la formula es=20 y el M del algoritmo es=13
Para t=1: el M de la formula es=20, y el M del algoritmo es=19
Para t=5: el M de la formula es=57, y el M del algoritmo es=56
Matriz P(0.5) con el algoritmo:
[[0.2506 0.217  0.3867 0.1458]
 [0.2531 0.2384 0.3744 0.1341]
 [0.1691 0.1936 0.4203 0.217 ]
 [0.158  0.1574 0.3983 0.2862]]
Matriz P(1) con el algoritmo:
[[0.2062 0.2039 0.3987 0.1912]
 [0.2083 0.2053 0.3979 0.1885]
 [0.1968 0.1984 0.401  0.2039]
 [0.192  0.194  0.4015 0.2125]]
Matriz P(5) con el algoritmo:
[[0.2 0.2 0.4 0.2]
 [0.2 0.2 0.4 0.2]
 [0.2 0.2 0.4 0.2]
 [0.2 0.2 0.4 0.2]]


Comparación de los resultados

Al comparar los dos métodos, vi que los resultados finales son idénticos en ambos casos. Esto demuestra que aunque se calculen de forma distinta, al final los dos nos dan la misma respuesta.

La principal diferencia radica en el número de pasos $M$ requeridos. Para $t=0.5$, el segundo método resulta notablemente más eficiente, ya que reduce el número de pasos de $20$ a $13$. En los casos $t=1$ y $t=5$, la mejora es menor, pues únicamente se reduce a un paso respecto al primer metodo.

Básicamente, el segundo método es mejor porque no hace cuentas de más; se detiene en cuanto tiene la respuesta, mientras que el primero sigue trabajando aunque ya no sea necesario.